isbuplicate

In [1]:
import pandas as pd
df = pd.read_csv("../data/NewData.csv")

print("rows:", len(df))
print("unique user_id:", df.user_id.nunique())
print("unique recording_id:", df.recording_id.nunique())

dup = df[df.duplicated("user_id", keep=False)].sort_values("user_id")
print("rows sharing a user_id:", len(dup))
print(dup[["user_id", "recording_id", "duration"]].head(20))

# files lost by the old naming (<user_id>_audio.wav)
print("recordings lost to collisions:", len(df) - df.user_id.nunique())

rows: 104
unique user_id: 102
unique recording_id: 104
rows sharing a user_id: 4
    user_id  recording_id  duration
3     93626        990175       475
73    93626        238079       478
21   147403        270150       584
25   147403        255381       780
recordings lost to collisions: 2


duration

In [2]:
import pandas as pd
df = pd.read_csv("../data/NewData.csv")
print("rows:", len(df))
print("unique recording_id:", df.recording_id.nunique())
print("unique user_id:", df.user_id.nunique())
print("duplicate rows:", df.duplicated().sum())
print("total hours:", df.duration.sum() / 3600)
print("rows with missing duration:", df.duration.isna().sum())

rows: 104
unique recording_id: 104
unique user_id: 102
duplicate rows: 0
total hours: 21.886111111111113
rows with missing duration: 0


pairing


In [3]:
import pandas as pd
df = pd.read_csv("../data/NewData.csv")
df["folder"] = df.rec_url_gcp.str.extract(r"upload_goai/(\d+)/")[0]
g = df.groupby("folder").agg(n=("recording_id", "size"),
                             users=("user_id", list),
                             durs=("duration", list))
print(g.n.value_counts())
print(g[g.n != 2])

n
2    52
Name: count, dtype: int64
Empty DataFrame
Columns: [n, users, durs]
Index: []


In [4]:
parent = {}
def find(x):
    parent.setdefault(x, x)
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

for _, r in df.iterrows():
    parent[find(f"f{r.folder}")] = find(f"u{r.user_id}")

df["group"] = [find(f"f{f}") for f in df.folder]
print("groups:", df.group.nunique())   # expect 50 (52 folders, 2 merges)
# df['group'].head()

groups: 50


In [5]:
import json, glob
tot, n = 0, 0
for f in glob.glob("../data/transcripts/*.json"):
    n += 1
    for s in json.load(open(f, encoding="utf-8")):
        tot += s["end"] - s["start"]
print("files:", n, "| transcribed speech hours:", tot / 3600)

files: 102 | transcribed speech hours: 12.113608333333316


In [6]:
import json, glob, os, wave
import numpy as np, pandas as pd

df = pd.read_csv("../data/NewData.csv")

def dur(path):
    with wave.open(path) as w:
        return w.getnframes() / w.getframerate(), w.getframerate(), w.getnchannels()

rows = []
for uid in sorted(df.user_id.unique()):
    p = f"../data/audios/{uid}_audio.wav"
    if not os.path.exists(p):
        print("missing audio:", uid); continue
    d, sr, ch = dur(p)
    cand = df[df.user_id == uid].copy()
    cand["diff"] = (cand.duration - d).abs()
    best = cand.sort_values("diff").iloc[0]
    if len(cand) > 1:
        print("COLLISION", uid, "audio s:", round(d), "csv:", cand.duration.tolist(), "-> picked", best.recording_id)
    rows.append((uid, best.recording_id, d, sr, ch, best["diff"]))

m = pd.DataFrame(rows, columns=["user_id","recording_id","audio_s","sr","ch","dur_diff"])
print(m.sr.value_counts(), m.ch.value_counts())
print("max duration mismatch vs CSV (s):", m.dur_diff.max())

lens, empty, n = [], 0, 0
for f in glob.glob("../data/transcripts/*.json"):
    for s in json.load(open(f, encoding="utf-8")):
        n += 1
        lens.append(s["end"] - s["start"])
        empty += not s["text"].strip()
lens = np.array(lens)
print("segments:", n, "| empty text:", empty)
print("seg length percentiles 5/50/95/max:", np.percentile(lens,[5,50,95]).round(2), lens.max().round(2))
print("segments > 30 s:", (lens > 30).sum())

COLLISION 93626 audio s: 401 csv: [475, 478] -> picked 990175
COLLISION 147403 audio s: 538 csv: [584, 780] -> picked 270150
sr
44100    102
Name: count, dtype: int64 ch
1    102
Name: count, dtype: int64
max duration mismatch vs CSV (s): 134.16573696145127
segments: 5857 | empty text: 0
seg length percentiles 5/50/95/max: [ 0.48  6.69 14.76] 15.0
segments > 30 s: 0


In [7]:
for i, r in m.iterrows():
    segs = json.load(open(f"../data/transcripts/{int(r.user_id)}_transcript.json", encoding="utf-8"))
    m.loc[i, "last_end"] = max(s["end"] for s in segs)
m["over"] = m.last_end - m.audio_s
print("files where transcript ends after audio:", (m.over > 1).sum())
print("audio hours (102 files):", m.audio_s.sum() / 3600)
print(m.sort_values("dur_diff", ascending=False).head(8)[["user_id", "audio_s", "last_end", "dur_diff"]])

files where transcript ends after audio: 0
audio hours (102 files): 20.037928162005542
    user_id     audio_s  last_end    dur_diff
44   213479  445.834263    438.14  134.165737
30   138962  689.015306    686.73  105.984694
23   100370  689.309206    689.31  105.690794
50   216926  474.615079    468.14  105.384921
26   106218  750.561179    742.50   95.438821
94   331421  751.464218    748.17   94.535782
72   246004  392.981338    392.00   82.018662
7     25489  563.650567    563.65   78.349433


In [8]:
import json, glob, random
random.seed(0)
segs = []
for f in glob.glob("../data/transcripts/*.json"):
    segs += [s["text"] for s in json.load(open(f, encoding="utf-8"))]
for t in random.sample(segs, 40):
    print(repr(t))

'जी'
'तो मैं अगर स्विच करना भी चाहूं तो कंप्यूटर साइंस में स्विच कर सकती हूं अ मुझे बस कुछ चीजें सीखनी होंगी कोडिंग वगैरा मैं सीख लूं मुझे चीजें बनानी आ जाए तो मैं कंप्यूटर लाइन में स्विच कर सकती हूं आराम से तो अगर मेरे को कल को'
'बोल सकते'
'हम्म, हार जीत का तो सबको अच्छा लगता है'
'ह्म्म ह्म्म बिल्कुलह्म्म'
'वहाँ पे अगर यह भी काम हो सकता था कि भईया तुम भागो जो तेज भागेगा वो बचेगा'
'हम्म लेकिन आज के टाइम पे जो लोग हैं ना वो कहीं ना कहीं देखा जाए तो स्वार्थी होते जा रहे हैं'
'हम्ममम'
'वो क्या फन सिटी होता है ना। वाटरपार्क वह सब मतलब जाता नहीं हूंँ, लेकिन अच्छा लगता है।'
'अपना मतलब स्कूल का एक्सपीरियंस सेयर करते हैं कि आज हमारा स्कूल में मतलब हमने क्या सीखा है होमबक मतलब क्या है ये सब चीज के बारे में डिस्कस करते हैं।'
'ऐसा तो है ही नही मुझे नहीं लगता कि ऐसा कुछ है कि हमसे गलती हो गई फिर हम उसे सुधार नहीं सकते, जो गलतियां हमसे होती हैं, उसका कोई न कोई रिकवरी का चांस रहता है, अब हमारे ये हमारे ऊपर है'
'की'
'उ हूं'
'वो कहते है ना जिस चीज़ को आप चाहते हो उसको उसको करियर बनाओ वो तो हाँ वो हो ज

In [9]:
import json, glob, re, collections
segs = []
for f in glob.glob("../data/transcripts/*.json"):
    segs += [s["text"] for s in json.load(open(f, encoding="utf-8"))]

print("with Latin letters:", sum(bool(re.search(r"[A-Za-z]", t)) for t in segs))
print("with digits:", sum(bool(re.search(r"\d", t)) for t in segs))
print("'ब्लैंक' segments:", sum(t.strip() == "ब्लैंक" for t in segs))

chars = collections.Counter(c for t in segs for c in t
                            if not ("\u0900" <= c <= "\u097F" or c == " "))
print(chars.most_common(25))
print([t for t in segs if re.search(r"[A-Za-z\d]", t)][:10])

with Latin letters: 198
with digits: 8
'ब्लैंक' segments: 5
[(',', 1125), ('E', 396), ('D', 396), ('R', 198), ('A', 198), ('C', 198), ('T', 198), ('.', 171), ('_', 33), ('!', 15), ('-', 14), ('0', 7), ('9', 5), ('1', 3), ('6', 3), ('4', 3), ('2', 2), ('5', 2), ('8', 1), (':', 1), ('7', 1)]
['REDACTED', 'REDACTED', 'REDACTED', 'REDACTED', 'REDACTED', 'REDACTED', 'REDACTED', 'REDACTED', 'REDACTED', 'REDACTED']


In [10]:
import json, glob, re
segs = []
for f in glob.glob("../data/transcripts/*.json"):
    segs += json.load(open(f, encoding="utf-8"))

red = [s for s in segs if "REDACTED" in s["text"]]
print("REDACTED segments:", len(red),
      "| exactly 'REDACTED':", sum(s["text"].strip() == "REDACTED" for s in red),
      "| hours:", round(sum(s["end"] - s["start"] for s in red) / 3600, 3))

def show(label, pat):
    hits = [s["text"] for s in segs if re.search(pat, s["text"]) and "REDACTED" not in s["text"]]
    print(f"\n{label}: {len(hits)}")
    for t in hits[:8]:
        print(repr(t))


show("underscore", r"_")
show("digits", r"\d")
show("dash", r"-")
show("danda", r"[।॥]")
show("devanagari digits", r"[०-९]")

# gpt
show("comma", r",")
show("period", r"\.")
show("exclamation", r"!")
show("colon", r":")
show("danda", r"[।॥]")

REDACTED segments: 198 | exactly 'REDACTED': 198 | hours: 0.43

underscore: 17
'और अगर हमने पहले से ही अ कही पे इंटर्नशिप कर रखी है तो वो काफी हमारे सी_वी को इनहांस करता है और हमें वो काम समझना और ज़्यादा आसान हो जाता है तो मेरा यह मानना है जी'
'कि अपने लिए अपना काम मैं खुद से करूं सारी चीजें अपने रेड्डी करके रखूं जिसे ना टी_वी वगैरह में निकदे आदमी जो लोग होते हैं उनको चइये उनके बी_वी बनाके दें उनके कपड़े निकाल के रखे उनके वाॅच वाॅलेट सब तो ये सारे चीजें मेंको पसंद नई हैं ज्यादा'
'और दूसरा भी एक आर्टिफिशियल इंटेलिजेंस आया है, ए_आई जो कि व्हाट्सएप वगैराह पे भी अवेलेबल है उसका सेक्टर तो उस पे मैंने जब अपने प्रश्नों को लिखा, तो उसका बहुत बेहतरीन ढंग से जो है उत्तर आया है, कि इन सब,'
'अच्छा आई_के_जी_ए आई कराए ओह'
'सूट पे दुपट्टे के जगहे साडी वेयर करते हैं ऐसी एकले लैगिंग के उपर साडी वेयर करना ये सब डी_आई_वाई है लोगों को पसंद आता है एक बार एक्चुली मैने खुदने भी कुर्ते के साथ साडी वेयर करी थी काफी अच्छा लुक ला आता है'
'इसलिए उतना इंटरेस्ट नहीं है और यहां पे इंडिया में होता भी नहीं है ओलंपिक 

In [11]:
%pip install soundfile scipy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import json, os, re, unicodedata
import numpy as np, pandas as pd, soundfile as sf
from scipy.signal import resample_poly

df = pd.read_csv("../data/NewData.csv")
df["folder"] = df.rec_url_gcp.str.extract(r"upload_goai/(\d+)/")[0]

# groups: rows sharing a folder or a user_id stay together
parent = {}
def find(x):
    parent.setdefault(x, x)
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
for _, r in df.iterrows():
    parent[find(f"f{r.folder}")] = find(f"u{r.user_id}")
group_of = {u: find(f"u{u}") for u in df.user_id.unique()}

def clean(t):
    t = unicodedata.normalize("NFC", t).replace("_", " ")
    t = re.sub(r"[,.।!:\-]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

OUT = "../data_clean"
os.makedirs(f"{OUT}/clips", exist_ok=True)
SR = 16000
rows, dropped_red, dropped_empty, dropped_zero = [], 0, 0, 0

for uid in sorted(group_of):
    audio, sr = sf.read(f"../data/audios/{uid}_audio.wav", dtype="float32")
    assert sr == 44100
    audio = resample_poly(audio, 160, 441).astype("float32")   # 44.1k -> 16k
    segs = json.load(open(f"../data/transcripts/{uid}_transcript.json", encoding="utf-8"))
    for i, s in enumerate(segs):
        if "REDACTED" in s["text"]:
            dropped_red += 1; continue
        text = clean(s["text"])
        if not text:
            dropped_empty += 1; continue
        a, b = int(s["start"] * SR), min(int(s["end"] * SR), len(audio))
        if b <= a:
            dropped_zero += 1; continue
        name = f"{uid}_{i:04d}.wav"
        sf.write(f"{OUT}/clips/{name}", audio[a:b], SR, subtype="PCM_16")
        rows.append(dict(clip=name, text=text, text_raw=s["text"],
                         duration=(b - a) / SR, user_id=uid, group=group_of[uid]))

m = pd.DataFrame(rows)

# split by group: 5 of 50 groups to validation
rng = np.random.default_rng(42)
groups = np.array(sorted(m.group.unique()))
val_groups = set(rng.choice(groups, size=round(len(groups) * 0.1), replace=False))
m["split"] = np.where(m.group.isin(val_groups), "val", "train")
m.to_csv(f"{OUT}/manifest.csv", index=False, encoding="utf-8-sig")

print("dropped REDACTED / empty-after-clean / zero-length:", dropped_red, dropped_empty, dropped_zero)
print("clips:", len(m), "| hours:", round(m.duration.sum() / 3600, 2))
print(m.groupby("split").agg(clips=("clip", "size"), hours=("duration", lambda x: round(x.sum() / 3600, 2)),
                             speakers=("user_id", "nunique"), groups=("group", "nunique")))
print("val/train speaker overlap:", set(m[m.split == "val"].user_id) & set(m[m.split == "train"].user_id))
cps = m.text.str.len() / m.duration
print("chars/sec percentiles 1/50/99:", np.percentile(cps, [1, 50, 99]).round(1))